# Plot Inference Results with Different Sample Sizes

Run `../script/run_1.sh` first. This generates inference results for synthetic datasets with different sample sizes, where sample size is the number of NK cells included in the analysis.

The goal is to assess how many NK cells are needed to recover the input parameters with reasonable accuracy.

This notebook loads the saved inference results from `../results/part_1/` and makes plots.

In [ ]:
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

import seaborn as sns
from matplotlib import gridspec
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import MultipleLocator, MaxNLocator, NullLocator

try:
    import scienceplots
    plt.style.use("science")
except ImportError:
    pass

RESULTS_DIR = Path("../results/part_1")

PARAM_LABELS = {
    "mu_lambda": r"$\mu_\lambda$",
    "sigma_lambda": r"$\sigma_\lambda$",
    "p_zero": r"$\phi_0$",
}

PARAM_XLIMS = {
    "mu_lambda": (0, 8),
    "sigma_lambda": (0, 6),
    "p_zero": (0, 1),
}

MODEL_NAME = "hetero3"
PARAMETERS = ("mu_lambda", "sigma_lambda", "p_zero")


In [9]:
def apply_param_ticks(
    ax,
    *,
    xparam: Optional[str] = None,
    yparam: Optional[str] = None,
    param_ticks: Optional[Dict[str, Sequence[float]]] = None,
    param_ticklabels: Optional[Dict[str, Sequence[str]]] = None,
) -> None:
    if not param_ticks:
        return

    if xparam in param_ticks:
        ticks = list(param_ticks[xparam])
        ax.set_xticks(ticks)
        if param_ticklabels and xparam in param_ticklabels:
            ax.set_xticklabels(list(param_ticklabels[xparam]))

    if yparam in param_ticks:
        ticks = list(param_ticks[yparam])
        ax.set_yticks(ticks)
        if param_ticklabels and yparam in param_ticklabels:
            ax.set_yticklabels(list(param_ticklabels[yparam]))

def plot_posteriors(
    idatas: Sequence[Tuple[str, az.InferenceData]],
    *,
    ground_truth: Optional[Dict[str, Dict[str, float]]] = None,
    parameters: Sequence[str],
    parameter_display: Optional[Dict[str, str]] = None,
    show_legend: bool = True,
    hdi_prob: float = 0.95,
    sample_size: int = 200000,
    save_path: str | Path = "posteriors",
    cmap_name: str = "inferno",
    font_scale: float = 0.7,
    diagonal_style: str = "hist",
    marginal_style: str = "circle",
    seed: Optional[int] = None,
    dpi: int = 300,
    xlims: Optional[Dict[str, Tuple[float, float]]] = None,
    label_size: int = 18,
    tick_size: int = 14,
    param_ticks: Optional[Dict[str, Sequence[float]]] = None,
    param_ticklabels: Optional[Dict[str, Sequence[str]]] = None,
) -> None:
    sns.set_context("talk", font_scale=float(font_scale))
    if len(idatas) == 1:
        colors = ["black"]
    else:
        cmap = plt.colormaps.get_cmap(str(cmap_name))
        colors = cmap(np.linspace(0.3, 0.9, len(idatas)))
    rng = np.random.default_rng(seed)

    params = [str(p) for p in parameters]
    if parameter_display is None:
        parameter_display = {p: p for p in params}

    def _posterior_vals(posterior, name: str) -> np.ndarray:
        if name in posterior:
            vals = posterior[name].stack(sample=("chain", "draw")).values.ravel()
            vals = np.asarray(vals, dtype=float)
            return vals[np.isfinite(vals)]
        return np.array([], dtype=float)

    label_to_df: Dict[str, pd.DataFrame] = {}
    for label, idata in idatas:
        posterior = idata.posterior
        df = pd.DataFrame()
        for p in params:
            vals = _posterior_vals(posterior, p)
            if vals.size == 0:
                continue
            if vals.size > int(sample_size):
                idx = rng.choice(vals.size, int(sample_size), replace=False)
                vals = vals[idx]
            df[p] = vals
        if not df.empty:
            df["label"] = label
            label_to_df[label] = df

    def _robust_limits(all_vals: np.ndarray) -> Optional[Tuple[float, float]]:
        all_vals = np.asarray(all_vals, dtype=float)
        all_vals = all_vals[np.isfinite(all_vals)]
        if all_vals.size == 0:
            return None
        lo = float(np.quantile(all_vals, 0.005))
        hi = float(np.quantile(all_vals, 0.995))
        if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
            lo = float(np.min(all_vals))
            hi = float(np.max(all_vals))
        if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
            return None
        pad = 0.03 * (hi - lo)
        return lo - pad, hi + pad

    param_xlims: Dict[str, Tuple[float, float]] = {}
    for p in params:
        cols: List[np.ndarray] = []
        for _lab, _df in label_to_df.items():
            if p in _df.columns:
                arr = np.asarray(_df[p].values, dtype=float)
                arr = arr[np.isfinite(arr)]
                if arr.size:
                    cols.append(arr)
        if cols:
            lims = _robust_limits(np.concatenate(cols))
            if lims is not None:
                param_xlims[p] = lims

    if xlims:
        for k, v in xlims.items():
            if k in params and v is not None:
                param_xlims[k] = (float(v[0]), float(v[1]))

    npar = len(params)
    fig = plt.figure(figsize=(5 * npar, 5 * npar), dpi=int(dpi))
    fig.patch.set_alpha(0.0)
    gs = gridspec.GridSpec(npar, npar, wspace=0.25, hspace=0.25)
    gaxes = np.empty((npar, npar), dtype=object)

    for irow, rowpar in enumerate(params):
        for icol, colpar in enumerate(params):
            ax = plt.subplot(gs[irow, icol])
            gaxes[irow, icol] = ax
            ax.set_facecolor("none")
            ax.xaxis.set_minor_locator(NullLocator())
            ax.yaxis.set_minor_locator(NullLocator())

            if icol > irow:
                ax.axis("off")
                continue

            for color, (label, df) in zip(colors, label_to_df.items()):
                if icol == irow:
                    if rowpar not in df.columns:
                        continue
                    vals = df[rowpar].dropna().values
                    if vals.size == 0:
                        continue

                    if diagonal_style == "kde":
                        sns.kdeplot(
                            vals,
                            ax=ax,
                            fill=True,
                            color=color,
                            alpha=0.2,
                            linewidth=1.5,
                            label=(label if (bool(show_legend) and irow == 0) else None),
                        )
                    else:
                        sns.histplot(vals, bins=30, stat="density", kde=False, ax=ax, color=color, alpha=0.18, element="step", fill=True)
                        sns.histplot(
                            vals,
                            bins=30,
                            stat="density",
                            kde=False,
                            ax=ax,
                            color=color,
                            alpha=1.0,
                            element="step",
                            fill=False,
                            linewidth=1.8,
                            label=(label if (bool(show_legend) and irow == 0) else None),
                        )

                    try:
                        lo, hi = az.hdi(vals, hdi_prob=float(hdi_prob))
                        ax.axvspan(float(lo), float(hi), color=color, alpha=0.1, linewidth=0)
                    except Exception:
                        pass

                    if ground_truth is not None and label in ground_truth and rowpar in ground_truth[label]:
                        ax.axvline(float(ground_truth[label][rowpar]), color=color, linestyle="-", linewidth=1.8)

                    ax.grid(alpha=0.2)
                else:
                    if (colpar not in df.columns) or (rowpar not in df.columns):
                        continue
                    if marginal_style == "circle":
                        sns.kdeplot(x=df[colpar], y=df[rowpar], ax=ax, fill=False, color=color, alpha=0.6, levels=7, linewidths=1.0)
                    else:
                        sns.histplot(x=df[colpar], y=df[rowpar], bins=60, pthresh=0.01, cmap=str(cmap_name), cbar=False, ax=ax)

            if icol != irow:
                ax.grid(alpha=0.3)
                if ground_truth is not None:
                    for color, (label, _df) in zip(colors, label_to_df.items()):
                        gt = ground_truth.get(label)
                        if gt and (colpar in gt) and (rowpar in gt):
                            ax.scatter(gt[colpar], gt[rowpar], marker="*", color=color, s=80, linewidths=2.0, zorder=1000)

            if icol == irow:
                ax.set_xlabel(parameter_display.get(rowpar, rowpar), fontsize=label_size)
                ax.set_ylabel("Density", fontsize=label_size)
            else:
                ax.set_xlabel(parameter_display.get(colpar, colpar), fontsize=label_size)
                ax.set_ylabel(parameter_display.get(rowpar, rowpar), fontsize=label_size)
            ax.tick_params(axis="both", which="major", labelsize=tick_size)
            # ax.tick_params(axis="both", which="minor", labelsize=tick_size)
            if icol == irow:
                xpar = rowpar
                if xpar in param_xlims:
                    ax.set_xlim(*param_xlims[xpar])
                apply_param_ticks(
                    ax,
                    xparam=rowpar,
                    param_ticks=param_ticks,
                    param_ticklabels=param_ticklabels,
                )
            else:
                if colpar in param_xlims:
                    ax.set_xlim(*param_xlims[colpar])
                if rowpar in param_xlims:
                    ax.set_ylim(*param_xlims[rowpar])
                apply_param_ticks(
                    ax,
                    xparam=colpar,
                    yparam=rowpar,
                    param_ticks=param_ticks,
                    param_ticklabels=param_ticklabels,
                )

            ax.margins(0.05)

    if bool(show_legend):
        handles, leg_labels = gaxes[0, 0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, leg_labels, loc="center", bbox_to_anchor=(0.85, 0.85), frameon=True, edgecolor="black", fontsize=10)
    
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path.with_suffix(".svg"), dpi=int(dpi), bbox_inches="tight", transparent=True)
    plt.close(fig)
    # print("Saved joint posterior plot:", str(save_path.with_suffix(".pdf")))


In [2]:
def load_config(n_cells: int, results_root: Path = RESULTS_DIR) -> dict:
    path = results_root / str(n_cells) / "config_ZI-gamma.json"
    return pd.read_json(path, typ="series").to_dict()

def load_posterior(n_cells: int, results_root: Path = RESULTS_DIR) -> az.InferenceData:
    path = results_root / str(n_cells) / "posterior_ZI-gamma_hetero3_smc.nc"
    return az.from_netcdf(path)

def load_logml(n_cells: int, results_root: Path = RESULTS_DIR) -> pd.DataFrame:
    path = results_root / str(n_cells) / "log_marginal_likelihood_ZI-gamma_hetero3_smc.csv"
    return pd.read_csv(path)


sample_sizes = sorted(
    int(p.name)
    for p in RESULTS_DIR.iterdir()
    if p.is_dir() and p.name.isdigit()
)

results = {
    n: {
        "config": load_config(n),
        "idata": load_posterior(n),
        "logml": load_logml(n),
    }
    for n in sample_sizes
}

print("sample size (number of in-silico NK cells) analysed:", "\n", sample_sizes)

sample size (number of in-silico NK cells) analysed: 
 [50, 100, 150, 200, 300, 400, 500, 600, 700, 750, 800]


In [12]:
FIG_DIR = Path("../figures/part_1")
FIG_DIR.mkdir(parents=True, exist_ok=True)

ground_truth = {}

phi_ticks = [0, 0.2, 0.4, 0.6, 0.8, 1.0]

for n in tqdm(sample_sizes):
    config = load_config(n)
    idata = load_posterior(n)

    label = f"n={n}"

    ground_truth[label] = {
        "mu_lambda": config["gt_mu_lambda"],
        "sigma_lambda": config["gt_sigma_lambda"],
        "p_zero": config["gt_p0_lambda"],
    }

    plot_posteriors(
        [(label, idata)],
        parameters=PARAMETERS,
        ground_truth=ground_truth,
        parameter_display=PARAM_LABELS,
        xlims=PARAM_XLIMS,
        show_legend=False,
        save_path=FIG_DIR / f"posterior_{label}.pdf",
        diagonal_style="hist",
        marginal_style="circle",
        label_size=26,
        tick_size=18,
        seed=2026,
        param_ticks={"p_zero": phi_ticks},
        param_ticklabels={"p_zero": ["0", "0.2", "0.4", "0.6", "0.8", "1.0"]},
    )

100%|██████████| 10/10 [05:31<00:00, 33.15s/it]


In [18]:
LABEL_SIZE = 22
TICK_SIZE = 18
LEGEND_SIZE = 20
TICK_NBINS = 4


def param_display_label(param: str) -> str:
    return PARAM_LABELS.get(param, param)


def posterior_values(idata: az.InferenceData, param: str) -> np.ndarray:
    if param not in idata.posterior:
        return np.array([], dtype=float)

    vals = idata.posterior[param].stack(sample=("chain", "draw")).values.ravel()
    vals = np.asarray(vals, dtype=float)
    return vals[np.isfinite(vals)]


def plot_hdi_summary(
    sample_sizes: Sequence[int],
    output_path: Path,
    *,
    results_root: Path = RESULTS_DIR,
    dpi: int = 350,
    hdi_prob: float = 0.95,
    cmap_name: str = "YlGnBu",
    cmap_min: float = 0.6,
    cmap_max: float = 0.95,
    xlim: Optional[Tuple[float, float]] = None,
) -> None:
    rows = []
    raw_samples: Dict[Tuple[str, int], np.ndarray] = {}

    for n_cells in sample_sizes:
        idata = load_posterior(n_cells, results_root)

        for param in PARAMETERS:
            vals = posterior_values(idata, param)
            if vals.size == 0:
                continue

            raw_samples[(param, int(n_cells))] = vals

            lo, hi = az.hdi(vals, hdi_prob=float(hdi_prob))
            rows.append(
                {
                    "n_cells": int(n_cells),
                    "param": param,
                    "hdi_low": float(lo),
                    "hdi_high": float(hi),
                }
            )

    if not rows:
        print("No posterior samples found.")
        return

    df = pd.DataFrame(rows).sort_values("n_cells")

    config0 = load_config(int(sample_sizes[0]), results_root)
    ground_truth = {
        "mu_lambda": float(config0["gt_mu_lambda"]),
        "sigma_lambda": float(config0["gt_sigma_lambda"]),
        "p_zero": float(config0["gt_p0_lambda"]),
    }

    cmap = plt.colormaps.get_cmap(str(cmap_name))
    colors = cmap(np.linspace(float(cmap_min), float(cmap_max), len(PARAMETERS)))
    param_to_color = {param: colors[i] for i, param in enumerate(PARAMETERS)}

    fig, axes = plt.subplots(
        len(PARAMETERS),
        1,
        figsize=(12, 14),
        dpi=int(dpi),
        sharex=True,
    )

    if len(PARAMETERS) == 1:
        axes = [axes]

    sample_sizes = [int(x) for x in sample_sizes]

    if len(sample_sizes) >= 2:
        min_gap = np.min(np.diff(sorted(sample_sizes)))
        violin_width = 0.7 * min_gap
    else:
        violin_width = 35

    for ax, param in zip(axes, PARAMETERS):
        ax.set_facecolor("none")

        sub = df[df["param"] == param].sort_values("n_cells")
        if sub.empty:
            ax.axis("off")
            continue

        color = param_to_color[param]

        x = sub["n_cells"].to_numpy(dtype=float)
        lo = sub["hdi_low"].to_numpy(dtype=float)
        hi = sub["hdi_high"].to_numpy(dtype=float)

        ax.fill_between(
            x,
            lo,
            hi,
            color=color,
            alpha=0.15,
            linewidth=0,
            label=f"{param_display_label(param)} {int(hdi_prob * 100)} \% HDI",
        )

        ax.plot(
            x,
            np.full_like(x, ground_truth[param], dtype=float),
            color=color,
            linestyle="--",
            linewidth=2.0,
            label=f"{param_display_label(param)} ground truth",
        )

        ax.grid(alpha=0.25)
        if xlim is not None:
            ax.set_xlim(*xlim)
        ax.tick_params(axis="both", which="major", labelsize=TICK_SIZE)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=TICK_NBINS))
        ax.set_ylabel(param_display_label(param), fontsize=LABEL_SIZE)
        ax.legend(frameon=False, fontsize=LEGEND_SIZE, loc="best")
        ax.xaxis.set_minor_locator(NullLocator())
        ax.yaxis.set_minor_locator(NullLocator())
        ax.minorticks_off()

    axes[-1].set_xlabel("Number of NK cells", fontsize=LABEL_SIZE)
    axes[-1].set_xticks(sample_sizes)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=int(dpi), bbox_inches="tight", transparent=True)
    plt.close(fig)

    print(f"Saved: {output_path}")

<>:112: SyntaxWarning: invalid escape sequence '\%'
<>:112: SyntaxWarning: invalid escape sequence '\%'
/var/folders/rr/16wv6ts1785fz6_2qhr5bc2m0000gn/T/ipykernel_87604/1770213741.py:112: SyntaxWarning: invalid escape sequence '\%'
  label=f"{param_display_label(param)} {int(hdi_prob * 100)} \% HDI",


In [19]:
FIG_DIR = Path("../figures/part_1")
FIG_DIR.mkdir(parents=True, exist_ok=True)

sample_sizes = sorted(
    int(p.name)
    for p in RESULTS_DIR.iterdir()
    if p.is_dir() and p.name.isdigit()
)

plot_hdi_summary(
    sample_sizes=sample_sizes,
    output_path=FIG_DIR / "hetero3_hdi_density_by_cell_number.svg",
    results_root=RESULTS_DIR,
    hdi_prob=0.95,
    xlim=(min(sample_sizes) - 25, max(sample_sizes) + 25),
)

Saved: ../figures/part_1/hetero3_hdi_density_by_cell_number.svg
